In [1]:
from PIL import Image
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import numpy as np

def extract_three_key_frames_safe(gif_path):
    """Extract 3 frames with robust error handling"""
    try:
        with Image.open(gif_path) as gif:
            # Count frames safely
            n_frames = 0
            max_frames = 200  # Safety limit to prevent memory issues
            
            try:
                while n_frames < max_frames:
                    gif.seek(n_frames)
                    n_frames += 1
            except EOFError:
                pass
            except Exception as e:
                # Some GIFs have weird errors mid-way
                if n_frames < 3:
                    return None, f"Frame counting failed: {str(e)[:50]}"
            
            if n_frames == 0:
                return None, "Empty GIF"
            
            if n_frames > max_frames:
                n_frames = max_frames  # Cap at safety limit
            
            # Calculate frame indices
            if n_frames < 3:
                indices = [0, 0, 0] if n_frames == 1 else [0, n_frames-1, n_frames-1]
            else:
                indices = [
                    n_frames // 4,      # 25%
                    n_frames // 2,      # 50%
                    3 * n_frames // 4   # 75%
                ]
            
            # Extract frames
            frames = []
            for idx in indices:
                try:
                    gif.seek(idx)
                    frame = gif.convert('RGB')
                    
                    # Validate frame
                    if frame.size[0] < 10 or frame.size[1] < 10:
                        return None, f"Frame too small: {frame.size}"
                    
                    frames.append(frame)
                    
                except Exception as e:
                    return None, f"Frame {idx} extraction failed: {str(e)[:50]}"
            
            if len(frames) != 3:
                return None, f"Got {len(frames)} frames instead of 3"
            
            return frames, None
            
    except Exception as e:
        return None, f"GIF open failed: {str(e)[:50]}"

def preprocess_multiframe_safe(csv_file, output_base_dir):
    """Process with detailed error tracking"""
    df = pd.read_csv(csv_file)
    
    split_name = Path(csv_file).stem.replace('_grouped', '')
    output_dir = Path(output_base_dir) / split_name
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"Processing {split_name} set: {len(df)} GIFs")
    print(f"{'='*60}")
    
    processed_data = []
    failed_gifs = []
    error_stats = {}
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=split_name):
        gif_path = row['gif_path']
        gif_id = Path(gif_path).stem
        
        # Extract frames
        frames, error = extract_three_key_frames_safe(gif_path)
        
        if frames is not None:
            try:
                # Save frames
                gif_dir = output_dir / gif_id
                gif_dir.mkdir(exist_ok=True)
                
                frame_paths = []
                for i, frame in enumerate(frames):
                    frame_path = gif_dir / f"frame_{i}.jpg"
                    frame.save(frame_path, quality=90)
                    frame_paths.append(str(frame_path))
                
                # Add to processed data
                row_data = row.to_dict()
                row_data['frame_path_0'] = frame_paths[0]
                row_data['frame_path_1'] = frame_paths[1]
                row_data['frame_path_2'] = frame_paths[2]
                processed_data.append(row_data)
                
            except Exception as e:
                error_msg = f"Save failed: {str(e)[:50]}"
                failed_gifs.append({'gif_id': gif_id, 'error': error_msg})
                error_stats[error_msg] = error_stats.get(error_msg, 0) + 1
        else:
            failed_gifs.append({'gif_id': gif_id, 'error': error})
            error_stats[error] = error_stats.get(error, 0) + 1
    
    # Save processed data
    processed_df = pd.DataFrame(processed_data)
    output_csv = output_base_dir / f"{split_name}_multiframe.csv"
    processed_df.to_csv(output_csv, index=False)
    
    # Save failed GIFs log
    if failed_gifs:
        failed_df = pd.DataFrame(failed_gifs)
        failed_csv = output_base_dir / f"{split_name}_failed.csv"
        failed_df.to_csv(failed_csv, index=False)
    
    # Print summary
    success_rate = (len(processed_df) / len(df)) * 100
    print(f"\n{'='*60}")
    print(f" {split_name} Results:")
    print(f"{'='*60}")
    print(f"  Successfully processed: {len(processed_df)}/{len(df)} ({success_rate:.1f}%)")
    print(f"  Failed: {len(failed_gifs)}")
    
    if error_stats:
        print(f"\n️ Error breakdown:")
        for error, count in sorted(error_stats.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"  {error[:60]}: {count}")
    
    if failed_gifs:
        print(f"\n Failed GIFs saved to: {failed_csv}")
    
    print(f"  Output: {output_csv}")
    
    return output_csv, processed_df, failed_gifs

# Run preprocessing with safety
OUTPUT_DIR = Path("processed_frames_multiframe")
OUTPUT_DIR.mkdir(exist_ok=True)

print(" MULTI-FRAME PREPROCESSING (SAFE MODE)")
print("Extracting 3 frames per GIF with crash protection\n")

train_csv, train_df, train_failed = preprocess_multiframe_safe('train_grouped.csv', OUTPUT_DIR)
val_csv, val_df, val_failed = preprocess_multiframe_safe('val_grouped.csv', OUTPUT_DIR)
test_csv, test_df, test_failed = preprocess_multiframe_safe('test_grouped.csv', OUTPUT_DIR)

print(f"\n{'='*60}")
print(" MULTI-FRAME PREPROCESSING COMPLETE!")
print(f"{'='*60}")
print(f"\nTotal processed:")
print(f"  Train: {len(train_df)} GIFs")
print(f"  Val:   {len(val_df)} GIFs")
print(f"  Test:  {len(test_df)} GIFs")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df)} GIFs")

print(f"\nTotal failed:")
print(f"  Train: {len(train_failed)}")
print(f"  Val:   {len(val_failed)}")
print(f"  Test:  {len(test_failed)}")

 MULTI-FRAME PREPROCESSING (SAFE MODE)
Extracting 3 frames per GIF with crash protection


Processing train set: 4145 GIFs


train: 100%|██████████| 4145/4145 [10:44<00:00,  6.43it/s]  



 train Results:
  Successfully processed: 4145/4145 (100.0%)
  Failed: 0
  Output: processed_frames_multiframe\train_multiframe.csv

Processing val set: 889 GIFs


val: 100%|██████████| 889/889 [02:27<00:00,  6.01it/s]



 val Results:
  Successfully processed: 889/889 (100.0%)
  Failed: 0
  Output: processed_frames_multiframe\val_multiframe.csv

Processing test set: 889 GIFs


test: 100%|██████████| 889/889 [02:02<00:00,  7.28it/s]


 test Results:
  Successfully processed: 889/889 (100.0%)
  Failed: 0
  Output: processed_frames_multiframe\test_multiframe.csv

 MULTI-FRAME PREPROCESSING COMPLETE!

Total processed:
  Train: 4145 GIFs
  Val:   889 GIFs
  Test:  889 GIFs
  Total: 5923 GIFs

Total failed:
  Train: 0
  Val:   0
  Test:  0


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

class MultiFrameEmotionDataset(Dataset):
    """Dataset that uses 3 frames per GIF"""
    
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform
        
        # Emotion mapping
        self.emotion_groups = sorted(self.data['emotion_group'].unique())
        self.group2idx = {group: idx for idx, group in enumerate(self.emotion_groups)}
        self.idx2group = {idx: group for group, idx in self.group2idx.items()}
        
        print(f" Loaded {len(self.data)} GIFs × 3 frames = {len(self.data)*3} total frames")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Load 3 frames
        frames = []
        for i in range(3):
            frame_path = row[f'frame_path_{i}']
            frame = Image.open(frame_path).convert('RGB')
            
            if self.transform:
                frame = self.transform(frame)
            
            frames.append(frame)
        
        # Stack: (3, C, H, W)
        frames_tensor = torch.stack(frames)
        
        # Label
        label = self.group2idx[row['emotion_group']]
        
        return frames_tensor, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = MultiFrameEmotionDataset('processed_frames_multiframe/train_multiframe.csv', transform=train_transform)
val_dataset = MultiFrameEmotionDataset('processed_frames_multiframe/val_multiframe.csv', transform=val_transform)
test_dataset = MultiFrameEmotionDataset('processed_frames_multiframe/test_multiframe.csv', transform=val_transform)

# DataLoaders
BATCH_SIZE = 16  # Reduced because we're processing 3x more data per sample

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n{'='*60}")
print(f" Multi-Frame DataLoaders Ready!")
print(f"{'='*60}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")
print(f"  Batch size: {BATCH_SIZE} GIFs × 3 frames")

print(f"\n Emotion Groups:")
for group, idx in sorted(train_dataset.group2idx.items(), key=lambda x: x[1]):
    count = len(train_dataset.data[train_dataset.data['emotion_group'] == group])
    print(f"  {idx}: {group:20s} ({count:4d} GIFs)")

# Test batch loading
print(f"\n Testing batch loading...")
import time
start = time.time()
batch_frames, batch_labels = next(iter(train_loader))
elapsed = time.time() - start

print(f" Batch loaded in {elapsed:.2f} seconds")
print(f"   Frames shape: {batch_frames.shape}")  # Should be (batch_size, 3, 3, 224, 224)
print(f"   Labels shape: {batch_labels.shape}")  # Should be (batch_size,)

 Loaded 4145 GIFs × 3 frames = 12435 total frames
 Loaded 889 GIFs × 3 frames = 2667 total frames
 Loaded 889 GIFs × 3 frames = 2667 total frames

 Multi-Frame DataLoaders Ready!
  Train batches: 260
  Val batches: 56
  Test batches: 56
  Batch size: 16 GIFs × 3 frames

 Emotion Groups:
  0: contempt             ( 123 GIFs)
  1: negative_intense     (1023 GIFs)
  2: negative_subdued     ( 684 GIFs)
  3: positive_calm        ( 681 GIFs)
  4: positive_energetic   (1262 GIFs)
  5: surprise             ( 372 GIFs)

 Testing batch loading...
 Batch loaded in 1.15 seconds
   Frames shape: torch.Size([16, 3, 3, 224, 224])
   Labels shape: torch.Size([16])


In [3]:
import torch.nn as nn
from torchvision import models

class MultiFrameEmotionClassifier(nn.Module):
    """
    Process 3 frames from each GIF and aggregate features
    Based on ResNet-ConvGRU paper approach
    """
    
    def __init__(self, num_classes=6, pretrained=True):
        super(MultiFrameEmotionClassifier, self).__init__()
        
        # Shared ResNet50 feature extractor for all frames
        resnet = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        
        # Remove final FC and avgpool
        self.features = nn.Sequential(*list(resnet.children())[:-2])  # Output: (batch, 2048, 7, 7)
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Frame aggregation: Concatenate features from 3 frames
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(2048 * 3, 1024),  # 3 frames concatenated
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # x shape: (batch, 3_frames, C, H, W)
        batch_size = x.size(0)
        num_frames = x.size(1)
        
        # Process each frame through shared CNN
        frame_features = []
        for i in range(num_frames):
            frame = x[:, i, :, :, :]  # (batch, C, H, W)
            
            # Extract features
            feat = self.features(frame)  # (batch, 2048, 7, 7)
            feat = self.avgpool(feat)    # (batch, 2048, 1, 1)
            feat = feat.view(batch_size, -1)  # (batch, 2048)
            
            frame_features.append(feat)
        
        # Concatenate features from all frames
        combined = torch.cat(frame_features, dim=1)  # (batch, 2048*3)
        
        # Classify
        output = self.classifier(combined)
        return output

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiFrameEmotionClassifier(num_classes=6, pretrained=True)
model = model.to(device)

print(f"\n{'='*60}")
print(f" Multi-Frame Model Created!")
print(f"{'='*60}")
print(f"  Device: {device}")
print(f"  Architecture: ResNet50 + Multi-Frame Aggregation")
print(f"  Input: 3 frames per GIF")
print(f"  Output: 6 emotion groups")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Test forward pass
test_input = torch.randn(2, 3, 3, 224, 224).to(device)  # 2 GIFs, 3 frames each
test_output = model(test_input)
print(f"\n Forward pass test:")
print(f"  Input: {test_input.shape}")
print(f"  Output: {test_output.shape}")
print(f"  Expected: (2, 6) " if test_output.shape == (2, 6) else "   Shape mismatch!")


 Multi-Frame Model Created!
  Device: cpu
  Architecture: ResNet50 + Multi-Frame Aggregation
  Input: 3 frames per GIF
  Output: 6 emotion groups
  Total parameters: 30,331,462
  Trainable parameters: 30,331,462

 Forward pass test:
  Input: torch.Size([2, 3, 3, 224, 224])
  Output: torch.Size([2, 6])
  Expected: (2, 6) 


In [4]:
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Class weights
emotion_labels = train_dataset.data['emotion_group'].map(train_dataset.group2idx).values
class_weights = compute_class_weight('balanced', classes=np.arange(6), y=emotion_labels)
class_weights = torch.FloatTensor(class_weights).to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print(f"\n{'='*60}")
print(f" Training Setup Complete!")
print(f"{'='*60}")
print(f"  Loss: CrossEntropyLoss (weighted)")
print(f"  Optimizer: Adam (lr=0.0001)")
print(f"  Scheduler: ReduceLROnPlateau")

print(f"\n Class Weights:")
for group, idx in sorted(train_dataset.group2idx.items(), key=lambda x: class_weights[x[1]], reverse=True):
    print(f"  {group:20s}: {class_weights[idx]:.3f}")


 Training Setup Complete!
  Loss: CrossEntropyLoss (weighted)
  Optimizer: Adam (lr=0.0001)
  Scheduler: ReduceLROnPlateau

 Class Weights:
  contempt            : 5.617
  surprise            : 1.857
  positive_calm       : 1.014
  negative_subdued    : 1.010
  negative_intense    : 0.675
  positive_energetic  : 0.547


In [5]:
from tqdm import tqdm

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for frames, labels in pbar:
        frames, labels = frames.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': f'{running_loss/len(loader):.4f}', 'acc': f'{100*correct/total:.2f}%'})
    
    return running_loss / len(loader), 100 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for frames, labels in tqdm(loader, desc='Validation'):
            frames, labels = frames.to(device), labels.to(device)
            outputs = model(frames)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return running_loss / len(loader), 100 * correct / total

print(" Training functions ready")

 Training functions ready


In [6]:
NUM_EPOCHS = 15
BEST_ACC = 0.0

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print(f"\n{'='*60}")
print(f" TRAINING MULTI-FRAME MODEL")
print(f"{'='*60}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {device}")
print(f"  Frames per GIF: 3")
print(f"  Training samples: {len(train_dataset)} GIFs")
print(f"{'='*60}\n")

for epoch in range(NUM_EPOCHS):
    print(f"\n Epoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"\n Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    if val_acc > BEST_ACC:
        BEST_ACC = val_acc
        torch.save(model.state_dict(), 'best_model_multiframe.pth')
        print(f"   New best model saved! (Val Acc: {val_acc:.2f}%)")

print(f"\n{'='*60}")
print(f" TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"  Best Validation Accuracy: {BEST_ACC:.2f}%")
print(f"  Previous (single-frame): 30.48%")
print(f"  Improvement: +{BEST_ACC - 30.48:.2f}%")
print(f"  Model saved as: best_model_multiframe.pth")


 TRAINING MULTI-FRAME MODEL
  Epochs: 15
  Device: cpu
  Frames per GIF: 3
  Training samples: 4145 GIFs


 Epoch 1/15
------------------------------------------------------------


Training: 100%|█████████▉| 259/260 [1:17:18<00:17, 17.91s/it, loss=1.9440, acc=17.86%]


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 1024])

In [7]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Recreate DataLoaders with drop_last=True
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"{'='*60}")
print(f" DataLoaders Fixed!")
print(f"{'='*60}")
print(f"  Train batches: {len(train_loader)} (drop_last=True)")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Recreate model (fresh start)
class MultiFrameEmotionClassifier(nn.Module):
    def __init__(self, num_classes=6, pretrained=True):
        super(MultiFrameEmotionClassifier, self).__init__()
        
        from torchvision import models
        resnet = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # No BatchNorm to avoid issues
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(2048 * 3, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        num_frames = x.size(1)
        
        frame_features = []
        for i in range(num_frames):
            frame = x[:, i, :, :, :]
            feat = self.features(frame)
            feat = self.avgpool(feat)
            feat = feat.view(batch_size, -1)
            frame_features.append(feat)
        
        combined = torch.cat(frame_features, dim=1)
        output = self.classifier(combined)
        return output

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiFrameEmotionClassifier(num_classes=6, pretrained=True)
model = model.to(device)

print(f"\n Model created (without BatchNorm)")

# Class weights
emotion_labels = train_dataset.data['emotion_group'].map(train_dataset.group2idx).values
class_weights = compute_class_weight('balanced', classes=np.arange(6), y=emotion_labels)
class_weights = torch.FloatTensor(class_weights).to(device)

# Training setup
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print(f" Training setup complete")

# Test forward pass
test_batch = next(iter(train_loader))
test_frames, test_labels = test_batch
test_frames = test_frames.to(device)
test_output = model(test_frames)
print(f"\n Forward pass test successful!")
print(f"   Input: {test_frames.shape}")
print(f"   Output: {test_output.shape}")

 DataLoaders Fixed!
  Train batches: 259 (drop_last=True)
  Val batches: 56
  Test batches: 56

 Model created (without BatchNorm)
 Training setup complete

 Forward pass test successful!
   Input: torch.Size([16, 3, 3, 224, 224])
   Output: torch.Size([16, 6])


In [8]:
from tqdm import tqdm

NUM_EPOCHS = 15
BEST_ACC = 0.0

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print(f"\n{'='*60}")
print(f" TRAINING MULTI-FRAME MODEL (FIXED)")
print(f"{'='*60}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {device}")
print(f"  Training samples: {len(train_dataset)}")
print(f"{'='*60}\n")

for epoch in range(NUM_EPOCHS):
    print(f"\n Epoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"\n Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    if val_acc > BEST_ACC:
        BEST_ACC = val_acc
        torch.save(model.state_dict(), 'best_model_multiframe.pth')
        print(f"   New best model saved! (Val Acc: {val_acc:.2f}%)")

print(f"\n{'='*60}")
print(f" TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"  Best Validation Accuracy: {BEST_ACC:.2f}%")


 TRAINING MULTI-FRAME MODEL (FIXED)
  Epochs: 15
  Device: cpu
  Training samples: 4145


 Epoch 1/15
------------------------------------------------------------


Validation: 100%|██████████| 56/56 [04:47<00:00,  5.13s/it]



 Epoch 1 Summary:
  Train Loss: 1.8037 | Train Acc: 21.55%
  Val Loss:   1.9687 | Val Acc:   17.89%
   New best model saved! (Val Acc: 17.89%)

 Epoch 2/15
------------------------------------------------------------


Validation: 100%|██████████| 56/56 [04:29<00:00,  4.81s/it]



 Epoch 2 Summary:
  Train Loss: 1.7878 | Train Acc: 22.10%
  Val Loss:   1.8575 | Val Acc:   15.86%

 Epoch 3/15
------------------------------------------------------------


...


 Epoch 13 Summary:
  Train Loss: 1.6672 | Train Acc: 25.34%
  Val Loss:   1.8089 | Val Acc:   26.77%

 Epoch 14/15
------------------------------------------------------------


Validation: 100%|██████████| 56/56 [08:46<00:00,  9.41s/it]



 Epoch 14 Summary:
  Train Loss: 1.6220 | Train Acc: 26.64%
  Val Loss:   1.8633 | Val Acc:   27.67%

 Epoch 15/15
------------------------------------------------------------


Validation: 100%|██████████| 56/56 [08:47<00:00,  9.42s/it]


 Epoch 15 Summary:
  Train Loss: 1.6041 | Train Acc: 27.10%
  Val Loss:   1.8859 | Val Acc:   28.23%

 TRAINING COMPLETE!
  Best Validation Accuracy: 28.57%
